<div align="center" style="text-align: center; padding: 60px 0;">

<h1 style="text-align: center;">Monitoring glacier retreat and glacial lake growth in the Mount Everest massif with Sentinel-2 (2017–2025)</h1>

<br>

<h2 style="text-align: center;">Remote Sensing &amp; Computer Vision</h2>

<br><br>

<p style="text-align: center; font-size: 18px;"><b>Authors:</b> Luka Rondal &amp; Tanguy Mandrillon</p>

<p style="text-align: center; font-size: 18px;">2026</p>

</div>


## **Context**
Glaciers of the Everest massif (Khumbu, Rongbuk, Kangshung, Ngozumpa, 
Imja–Lhotse Shar...) are losing mass because of rising temperatures. Two visible 
consequences are the shrinking of clean ice/snow-covered surfaces and the growth of 
pro-glacial lakes such as Imja Tsho, which increases the risk of glacial lake outburst 
floods (GLOF) for the valleys downstream. The goal is to quantify these two phenomena 
over the last decade. 

## 1. Bibliography
In this section, we will define the scope and make sope assumption for our analysis.

### **Geography analysis**
We will chose a glacier and a lake to analyze.

| Type | Name | Description | Source |
|:---|:---|:---|:---|
| Glacier | Khumbu Glacier | Located in north-eastern Nepal, between Mount Everest and the Lhotse–Nuptse ridge. It is one of the highest glaciers in the world and includes the Khumbu Icefall on the standard Everest climbing route. Its lower tongue is largely debris-covered. | [Wikipedia – Khumbu Glacier](https://en.wikipedia.org/wiki/Khumbu_Glacier) |
| Lake | Imja Tsho | Pro-glacial lake at about 5,000 m altitude, at the terminus of the Imja and Lhotse Shar glaciers, south of Everest. It formed from melt ponds in the second half of the 20th century and is one of the fastest-growing glacial lakes in the Himalaya, with a significant GLOF risk for the valleys downstream. | [Wikipedia – Imja Tsho](https://en.wikipedia.org/wiki/Imja_Tsho) |

### **Spectral indices: NDSI and NDWI**

To separate our classes we use two **normalized difference indices**. A normalized difference index compares the reflectance of a pixel in two spectral bands where the target surface behaves very differently:

$$\text{Index} = \frac{\rho_1 - \rho_2}{\rho_1 + \rho_2} \in [-1, 1]$$

The normalization makes the index less sensitive to illumination differences (slopes, shadows, sun angle), which matters in a steep mountain area like the Everest massif.

**[NDSI – Normalized Difference Snow Index](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/ndsi/)**

$$NDSI = \frac{B03 - B11}{B03 + B11}$$

- `B03` = green (560 nm, 10 m) and `B11` = short-wave infrared, SWIR (1610 nm, 20 m).
- **What it shows:** snow and ice reflect a lot of visible light but absorb strongly in the SWIR, so they have a **high NDSI**. Rock, soil and debris reflect about the same amount in both bands, so their NDSI is **close to 0 or negative**. Clouds are bright in both bands, which gives a low NDSI and lets the index separate snow from most clouds.
- **Threshold:** a pixel is classified as snow/ice when **NDSI > 0.42** (Hall et al., 1995).
- **Limitation:** NDSI cannot tell snow from clean glacier ice (both are high), so we merge them into one **snow/ice** class. Ice covered by debris looks like rock and is **not** detected.

**[NDWI – Normalized Difference Water Index](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/ndwi/)**

$$NDWI = \frac{B03 - B08}{B03 + B08}$$

- `B03` = green (560 nm, 10 m) and `B08` = near infrared, NIR (842 nm, 10 m).
- **What it shows:** open water absorbs almost all NIR light, so it has a **positive NDWI**. Vegetation and bare soil reflect more NIR than green, so they have a **negative NDWI**.
- **Threshold:** the threshold depends on the source. McFeeters (1996) uses **NDWI > 0** for water, while Sentinel Hub states that *"index values greater than 0.5 usually correspond to water bodies"*. In a glacier area, a threshold that is too low is risky: snow and clean ice can also have slightly positive NDWI values. A threshold that is too high misses turbid glacial lakes, which carry a lot of sediment. We will therefore **tune the threshold between 0.2 and 0.5** by checking the result on true-colour images of Imja Tsho.
- **Why we need it:** water also has a **high NDSI**, since it absorbs SWIR as well. NDSI alone would classify lakes such as Imja Tsho as ice, so NDWI is used to **separate lakes from ice**.

### **Classification rules**

Each pixel gets one of 4 classes. The order of the tests matters:

| Order | Condition | Class |
|:---:|:---|:---|
| 1 | SCL = 0, 3, 8, 9 or 10 (no data, cloud shadow, clouds, cirrus) | Cloud / no data |
| 2 | NDWI > threshold (0.2 – 0.5, tuned) | Water (glacial lake) |
| 3 | NDSI > 0.4 | Snow / ice |
| 4 | otherwise | Rock / debris |

Water is tested **before** snow/ice because lakes have both a high NDWI and a high NDSI. Area per class = number of pixels × 400 m² (20 m pixels).

## 2. Parameters

All the **variable inputs** of the pipeline are gathered in the next cell. To run the analysis on another area, period or threshold, only this cell has to be modified, then the whole notebook can be re-run.

| Parameter | Value | Justification |
|:---|:---|:---|
| `BBOX_KHUMBU` | 86.79–86.94° E, 27.92–28.02° N | Khumbu Glacier, from its terminus (27.93° N, 86.81° E) up to the Khumbu Icefall and the Western Cwm |
| `BBOX_IMJA_TSHO` | 86.90–86.97° E, 27.88–27.92° N | Imja Tsho (27.90° N, 86.93° E, about 2 km × 650 m) and the tongues of the Imja and Lhotse Shar glaciers that feed it |
| `YEARS` | 2017 → 2025 | One image per year, to follow the trend over the last decade |
| `SEASON_START` / `SEASON_END` | 1 Oct → 30 Nov | Post-monsoon: fewest clouds and minimal seasonal snow, so the snow/ice surface is close to the glacier surface |
| `MAX_AOI_CLOUD` | 5 % | A date is a candidate only if less than 5 % of the area of interest is masked (clouds, shadows, no data) |
| `MAX_HAZE_RATIO` | 1.12 | Rejects hazy dates that the SCL does not detect (see section 3) |
| `SCREENING_RESOLUTION` | 60 m | Resolution used to compare all the dates of the window before choosing one |
| `RESOLUTION` | 20 m | Native resolution of B11 (SWIR); same grid for all the bands |
| `NDSI_THRESHOLD` | 0.42 | Snow/ice threshold (see section 1) |
| `NDWI_THRESHOLD` | 0.3 | Water threshold, starting value to tune between 0.2 and 0.5 (see section 1) |
| `SCL_MASK_CLASSES` | 0, 3, 8, 9, 10 | No data, cloud shadow, clouds (medium/high probability), thin cirrus |
| `MAX_MASKED_FRACTION` | 30 % | Years with more masked pixels are flagged as unreliable |

The Copernicus credentials are **not** written in the notebook: they are imported from the separate file `myCopernicusCredentials.py`.

In [ ]:
# --- Areas of interest (lon_min, lat_min, lon_max, lat_max) in WGS84 (EPSG:4326)
BBOX_KHUMBU = [86.79, 27.92, 86.94, 28.02]     # Khumbu Glacier: from the terminus (27.93 N, 86.81 E) to the Western Cwm
BBOX_IMJA_TSHO = [86.90, 27.88, 86.97, 27.92]  # Imja Tsho (27.90 N, 86.93 E) + Imja / Lhotse Shar glacier tongues

AOIS = {
    "Khumbu Glacier": BBOX_KHUMBU,
    "Imja Tsho": BBOX_IMJA_TSHO,
}

# --- Period
YEARS = list(range(2017, 2026))  # 2017 -> 2025 (range end is excluded)
SEASON_START = "10-01"           # MM-DD, post-monsoon window
SEASON_END = "11-30"             # MM-DD

# --- Image selection: for each year, the clear acquisition with the least snow (see section 3)
MAX_AOI_CLOUD = 0.05       # max fraction of masked pixels (SCL) on the area of interest for a candidate date
MAX_HAZE_RATIO = 1.12      # max haze indicator of a candidate date, relative to the median of the clear dates
SCREENING_RESOLUTION = 60  # pixel size (m) used to compare the candidate dates (fast and cheap)
RESOLUTION = 20         # Pixel size in meters (B11 native resolution)

# --- Classification thresholds (see section 1)
NDSI_THRESHOLD = 0.42  # snow/ice if NDSI > NDSI_THRESHOLD
NDWI_THRESHOLD = 0.3   # water if NDWI > NDWI_THRESHOLD (to tune between 0.2 and 0.5)

# --- Cloud / no-data mask from the Scene Classification Layer (SCL)
SCL_MASK_CLASSES = [0, 3, 8, 9, 10]  # no data, cloud shadow, cloud medium/high proba, thin cirrus
MAX_MASKED_FRACTION = 0.30           # years with more masked pixels are flagged

# --- Bands requested to Sentinel Hub (Sentinel-2 L2A)
BANDS = ["B02", "B03", "B04", "B08", "B11", "SCL"]

# --- Derived values
PIXEL_AREA_KM2 = (RESOLUTION * RESOLUTION) / 1e6  # 20 m x 20 m = 400 m2 = 0.0004 km2
TIME_INTERVALS = {year: (f"{year}-{SEASON_START}", f"{year}-{SEASON_END}") for year in YEARS}

for name, bbox in AOIS.items():
    print(f"{name}: {bbox}")
print(f"Years: {YEARS[0]} -> {YEARS[-1]} ({len(YEARS)} images)")
print(f"Season window: {SEASON_START} -> {SEASON_END}")
print(f"Pixel area: {PIXEL_AREA_KM2} km2")

Next we configure the connection to the Copernicus Data Space Ecosystem (Sentinel Hub API) and check that the requested image stays under the Sentinel Hub size limit (2500 × 2500 px).

In [13]:
import sys
import importlib
import sentinelhub
import myCopernicusCredentials
importlib.reload(myCopernicusCredentials)

# --- Sentinel Hub configuration (Copernicus Data Space Ecosystem)
config = sentinelhub.SHConfig()
config.sh_client_id = myCopernicusCredentials.client_id
config.sh_client_secret = myCopernicusCredentials.client_secret
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
config.sh_base_url = "https://sh.dataspace.copernicus.eu"

# --- Bounding boxes and image sizes
aoi_bboxes = {}
aoi_sizes = {}
for name, bbox in AOIS.items():
    aoi_bboxes[name] = sentinelhub.BBox(bbox=bbox, crs=sentinelhub.CRS.WGS84)
    aoi_sizes[name] = sentinelhub.bbox_to_dimensions(aoi_bboxes[name], resolution=RESOLUTION)
    print(f"{name}: {aoi_sizes[name][0]} x {aoi_sizes[name][1]} px at {RESOLUTION} m")

    if max(aoi_sizes[name]) > 2500:
        print(f"WARNING: {name} larger than 2500 px, increase RESOLUTION or reduce the bbox")

Khumbu Glacier: 738 x 553 px at 20 m
Imja Tsho: 345 x 221 px at 20 m


In [19]:
import folium
import shapely

# Visual check of the areas of interest
colors = {"Khumbu Glacier": "orange", "Imja Tsho": "red"}

# Base maps: OpenStreetMap by default, satellite imagery selectable in the legend
map = folium.Map(tiles="OpenStreetMap")
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                 attr="Esri World Imagery", name="Satellite (Esri)", show=False).add_to(map)

for name, bbox in AOIS.items():
    folium.GeoJson(data=shapely.geometry.box(*bbox),
                   name=f"<span style='color: {colors[name]};'>▬</span> {name}",
                   style_function=lambda x, c=colors[name]: {"color": c, "fillOpacity": 0.05}).add_to(map)

map.add_child(folium.LayerControl(collapsed=False))
# Fit the map on all the areas of interest. folium expects (lat, lon)
lon_min, lat_min = min(b[0] for b in AOIS.values()), min(b[1] for b in AOIS.values())
lon_max, lat_max = max(b[2] for b in AOIS.values()), max(b[3] for b in AOIS.values())
map.fit_bounds([(lat_min, lon_min), (lat_max, lon_max)])
display(map)

## 3. Data acquisition

For each year and each area of interest, we **select one Sentinel-2 L2A acquisition** in the post-monsoon window (1 Oct → 30 Nov), then download it.

**Why not simply `leastCC`?** The `leastCC` mosaicking keeps the acquisition whose Sentinel-2 *tile* (100 × 100 km) is the least cloudy. It ignores snow: a clear image taken just after a snowfall can be selected, and the fresh snow then inflates the snow/ice area of that year. We tested it: in 2024, `leastCC` selected 7 Nov, just after a snowfall (62 % snow/ice on the Khumbu area), while the clear images of 12–22 Nov give 44–47 %, like the other years.

**Selection rule: the clear image with the least snow.** Mapping glaciers on the image with the minimum snow cover at the end of the melt season is a classic approach: seasonal snow is then as low as possible and the snow/ice surface is closest to the glacier surface. For each year:
1. list all the acquisitions of the window with the **Catalog API**;
2. download each of them at `SCREENING_RESOLUTION` (60 m) and compute, on the area of interest, the **masked fraction** (SCL mask of section 4) and the **snow/ice fraction** (NDSI > `NDSI_THRESHOLD` and not water, section 1);
3. keep the dates fully acquired, with less than `MAX_AOI_CLOUD` of masked pixels and **without haze** (see below);
4. **select the date with the lowest snow/ice fraction**;
5. download this date only, at 20 m.

**Haze filter.** Thin haze is often not detected by the SCL, but it lowers the NDSI of snow: a hazy image then looks less snowy and would be selected by the rule (e.g. Khumbu, 29 Oct 2023). Haze brightens dark surfaces in the blue band, so we use as haze indicator the **median B02 reflectance of the non-snow pixels** (NDSI < 0). For each area, a date is rejected if this value is more than `MAX_HAZE_RATIO` times the median of all the clear dates. On our data, the clear dates are between 0.85 and 1.10 times the median, the hazy ones between 1.14 and 1.45.

**Request details:**
- **Evalscript:** returns the bands `B02, B03, B04, B08, B11` as **reflectance** (FLOAT32, 0 to ~1), the `SCL` class code and `dataMask` (1 = pixel acquired, 0 = no data). The raw values are kept so the indices can be computed in Python.
- **Resolution:** 20 m. B11 is natively at 20 m, the 10 m bands are resampled on the same grid.
- **Cache:** the responses are saved in `data/sentinelhub/`. Re-running the notebook reloads them from disk instead of downloading them again.

In [ ]:
import json
import numpy as np

# --- Bands returned by the evalscript (BANDS from the parameters + dataMask)
OUTPUT_BANDS = BANDS + ["dataMask"]
BAND_INDEX = {band: i for i, band in enumerate(OUTPUT_BANDS)}  # e.g. image[..., BAND_INDEX["B03"]]

# Reflectance for the spectral bands, raw class code (DN) for SCL and dataMask
units = ["DN" if band in ("SCL", "dataMask") else "REFLECTANCE" for band in OUTPUT_BANDS]

evalscript_bands = f"""
//VERSION=3
function setup() {{
    return {{
        input: [{{
            bands: {json.dumps(OUTPUT_BANDS)},
            units: {json.dumps(units)}
        }}],
        output: {{
            bands: {len(OUTPUT_BANDS)},
            sampleType: "FLOAT32"
        }}
    }};
}}

function evaluatePixel(sample) {{
    return [{", ".join(f"sample.{band}" for band in OUTPUT_BANDS)}];
}}
"""

# Sentinel-2 L2A collection on the Copernicus Data Space Ecosystem
S2_L2A = sentinelhub.DataCollection.SENTINEL2_L2A.define_from(name="s2l2a_cdse", service_url=config.sh_base_url)

DATA_FOLDER = "data/sentinelhub"  # local cache of the downloaded images


def download_image(bbox_sh, size, time_interval):
    """Download one Sentinel-2 L2A image (H x W x len(OUTPUT_BANDS), float32) for the time interval."""
    request = sentinelhub.SentinelHubRequest(
        evalscript=evalscript_bands,
        input_data=[
            sentinelhub.SentinelHubRequest.input_data(
                data_collection=S2_L2A,
                time_interval=time_interval,
                mosaicking_order=sentinelhub.MosaickingOrder.LEAST_CC,  # merges the products of a same day
            )
        ],
        responses=[sentinelhub.SentinelHubRequest.output_response("default", sentinelhub.MimeType.TIFF)],
        bbox=bbox_sh,
        size=size,  # WARNING, the output size should be < (2500, 2500)
        data_folder=DATA_FOLDER,
        config=config,
    )
    return request.get_data(save_data=True)[0]  # reloaded from DATA_FOLDER if already downloaded


def normalized_difference(band_a, band_b):
    """(a - b) / (a + b), with 0 where a + b = 0."""
    total = band_a + band_b
    return np.divide(band_a - band_b, total, out=np.zeros_like(total), where=total != 0)


def image_statistics(image):
    """Coverage, masked fraction, snow/ice fraction (on the valid pixels) and haze indicator of one image."""
    acquired = image[..., BAND_INDEX["dataMask"]] == 1
    masked = np.isin(image[..., BAND_INDEX["SCL"]], SCL_MASK_CLASSES) | ~acquired
    ndsi = normalized_difference(image[..., BAND_INDEX["B03"]], image[..., BAND_INDEX["B11"]])
    ndwi = normalized_difference(image[..., BAND_INDEX["B03"]], image[..., BAND_INDEX["B08"]])
    snow_ice = (ndsi > NDSI_THRESHOLD) & (ndwi <= NDWI_THRESHOLD) & ~masked
    non_snow = (ndsi < 0) & ~masked  # rock, debris, vegetation
    return {
        "coverage": acquired.mean(),
        "masked": masked.mean(),
        "snow_ice": snow_ice.sum() / max((~masked).sum(), 1),
        # Haze indicator: haze brightens the dark surfaces in the blue band
        "haze_blue": np.median(image[..., BAND_INDEX["B02"]][non_snow]) if non_snow.sum() > 50 else np.nan,
    }

### Image selection

We screen every acquisition of the window at 60 m. The table `candidates` lists all the dates with their coverage, masked and snow/ice fractions; `selected` keeps, for each year, the clear date without haze with the least snow/ice.

The first run sends one small request per date (about 10–15 dates per year and per area), so it takes a few minutes. The next runs reload them from the cache.

In [ ]:
import pandas as pd

catalog = sentinelhub.SentinelHubCatalog(config=config)

candidate_rows = []
for name in AOIS:
    screening_size = sentinelhub.bbox_to_dimensions(aoi_bboxes[name], resolution=SCREENING_RESOLUTION)
    for year, time_interval in TIME_INTERVALS.items():
        # 1. All the acquisitions of the window
        results = catalog.search(
            S2_L2A,
            bbox=aoi_bboxes[name],
            time=time_interval,
            fields={"include": ["properties.datetime"], "exclude": []},
        )
        dates = sorted({r["properties"]["datetime"][:10] for r in results})
        print(f"{name} {year}: {len(dates)} dates to screen")

        # 2. Statistics of each date at low resolution
        for date in dates:
            stats = image_statistics(download_image(aoi_bboxes[name], screening_size, (date, date)))
            candidate_rows.append({
                "aoi": name,
                "year": year,
                "date": date,
                "coverage_%": 100 * stats["coverage"],
                "masked_%": 100 * stats["masked"],
                "snow_ice_%": 100 * stats["snow_ice"],
                "haze_blue": stats["haze_blue"],
            })

candidates = pd.DataFrame(candidate_rows).round({"coverage_%": 1, "masked_%": 1, "snow_ice_%": 1, "haze_blue": 3})

# 3. Keep the clear dates without haze (haze relative to the median of the clear dates of the same area)
clear = candidates[(candidates["coverage_%"] >= 99) & (candidates["masked_%"] <= 100 * MAX_AOI_CLOUD)].copy()
clear["haze_ratio"] = (clear["haze_blue"] / clear.groupby("aoi")["haze_blue"].transform("median")).round(2)
clear = clear[clear["haze_ratio"] <= MAX_HAZE_RATIO]

# 4. Select the clear date with the least snow/ice
selected = clear.loc[clear.groupby(["aoi", "year"])["snow_ice_%"].idxmin()].set_index(["aoi", "year"])
display(selected)

for name in AOIS:
    missing = [year for year in YEARS if (name, year) not in selected.index]
    if missing:
        print(f"{name}: no clear date for {missing} (increase MAX_AOI_CLOUD or widen the window)")

We now download, at 20 m, only the selected date of each year: 2 areas × 9 years = 18 requests.

In [ ]:
# --- Download the selected date of each year and each area of interest
# images["Imja Tsho"][2020] -> numpy array (H x W x len(OUTPUT_BANDS))
images = {name: {} for name in AOIS}

for (name, year), row in selected.iterrows():
    date = row["date"]
    images[name][year] = download_image(aoi_bboxes[name], aoi_sizes[name], (date, date))
    print(f"{name} {year}: {date}, snow/ice = {row['snow_ice_%']} %, masked = {row['masked_%']} %")

### Image quality check

We display the true-colour composite (B04, B03, B02) of each year. For each image, check:
- **Clouds:** remaining clouds over the glaciers or the lake. They will be masked with the SCL in section 4.
- **Seasonal snow:** fresh snow down in the valleys makes the snow/ice area of that year too large and not comparable with the other years.
- **Shadows:** in October–November the sun is low, so the north faces are in deep shadow.

Years with obvious problems should be noted here and flagged in the analysis.

In [ ]:
import matplotlib.pyplot as plt


def true_color(image, gain=2.5):
    """RGB composite (B04, B03, B02) scaled to [0, 1] for display."""
    rgb = image[..., [BAND_INDEX["B04"], BAND_INDEX["B03"], BAND_INDEX["B02"]]]
    return np.clip(rgb * gain, 0, 1)


def show_true_color_grid(name, gain=2.5, ncols=3):
    """Display the true-colour image of every year for one area of interest."""
    years = list(images[name])
    nrows = int(np.ceil(len(years) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
    for ax in axes.flat:
        ax.axis("off")
    for ax, year in zip(axes.flat, years):
        ax.imshow(true_color(images[name][year], gain))
        ax.set_title(f"{name} - {selected.loc[(name, year), 'date']}")
    plt.tight_layout()
    plt.show()


# gain=1.5 keeps some detail on snow, which saturates quickly
for name in AOIS:
    show_true_color_grid(name, gain=1.5)

## 4. Pre-processing

Before computing the indices, we remove the pixels that cannot be trusted. We use the **Scene Classification Layer (SCL)** produced by Sen2Cor with each L2A image:

| SCL code | Meaning | Why it is masked |
|:---:|:---|:---|
| 0 | No data | Pixel not acquired |
| 3 | Cloud shadow | Dark pixel: NDSI/NDWI are unreliable |
| 8 | Cloud, medium probability | Clouds are bright in the visible and can look like snow |
| 9 | Cloud, high probability | Same |
| 10 | Thin cirrus | Alters the reflectance of the surface below |

Pixels with `dataMask = 0` are also masked. The masked pixels will form the **cloud / no data** class of the classification (section 5).

For each image we compute the **valid-pixel fraction**. A year with more than `MAX_MASKED_FRACTION` of masked pixels is **flagged**: its areas are not comparable with the other years, because part of the glacier or of the lake is hidden.

In [30]:
def compute_invalid_mask(image):
    """True where the pixel is masked: SCL in SCL_MASK_CLASSES or no data (dataMask = 0)."""
    scl = image[..., BAND_INDEX["SCL"]]
    return np.isin(scl, SCL_MASK_CLASSES) | (image[..., BAND_INDEX["dataMask"]] == 0)


def scl_fraction(image, classes):
    """Fraction of the pixels whose SCL code is in `classes`."""
    return np.isin(image[..., BAND_INDEX["SCL"]], classes).mean()


# masks["Imja Tsho"][2020] -> boolean array (H x W), True = masked pixel
masks = {name: {} for name in AOIS}
preprocessing_rows = []

for name in AOIS:
    for year, image in images[name].items():
        masks[name][year] = compute_invalid_mask(image)
        masked_fraction = masks[name][year].mean()

        preprocessing_rows.append({
            "aoi": name,
            "year": year,
            "no_data_%": 100 * (1 - image[..., BAND_INDEX["dataMask"]].mean()),
            "cloud_%": 100 * scl_fraction(image, [8, 9, 10]),
            "cloud_shadow_%": 100 * scl_fraction(image, [3]),
            "masked_%": 100 * masked_fraction,
            "valid_%": 100 * (1 - masked_fraction),
            "topographic_shadow_%": 100 * scl_fraction(image, [2]),  # SCL 2, not masked (for information)
            "flagged": masked_fraction > MAX_MASKED_FRACTION,
        })

preprocessing_summary = pd.DataFrame(preprocessing_rows).set_index(["aoi", "year"]).round(1)
display(preprocessing_summary)

flagged_years = preprocessing_summary[preprocessing_summary["flagged"]]
if flagged_years.empty:
    print(f"No year flagged: all images have less than {MAX_MASKED_FRACTION:.0%} of masked pixels.")
else:
    print("Flagged years (too many masked pixels):")
    display(flagged_years)

no_data_%  cloud_%  cloud_shadow_%  masked_%  valid_%  \
aoi            year                                                          
Khumbu Glacier 2017        0.0      0.2             0.0       0.2     99.8   
               2018        0.0      0.0             0.0       0.0    100.0   
               2019        0.0      0.0             0.0       0.0    100.0   
               2020        0.0      0.0             0.0       0.0    100.0   
               2021        0.0      0.4             0.0       0.4     99.6   
               2022        0.0      0.5             0.0       0.5     99.5   
               2023        0.0      0.0             0.0       0.0    100.0   
               2024        0.0      0.8             0.0       0.8     99.2   
               2025        0.0      0.0             0.0       0.0    100.0   
Imja Tsho      2017        0.0      0.0             0.0       0.0    100.0   
               2018        0.0      0.0             0.0       0.0    100.0   
               2019        0.0      0.1             0.0       0.1     99.9   
               2020        0.0      0.0             0.0       0.0    100.0   
               2021        0.0      0.2             0.0       0.2     99.8   
               2022        0.0      2.3             0.0       2.3     97.7   
               2023        0.0      1.1             0.0       1.1     98.9   
               2024        0.0      5.0             0.0       5.0     95.0   
               2025        0.0      0.2             0.0       0.2     99.8   

                     topographic_shadow_%  flagged  
aoi            year                                 
Khumbu Glacier 2017                  18.3    False  
               2018                   6.8    False  
               2019                  14.7    False  
               2020                  13.7    False  
               2021                  18.3    False  
               2022                  16.7    False  
               2023                  13.2    False  
               2024                  10.9    False  
               2025                  15.6    False  
Imja Tsho      2017                  23.4    False  
               2018                  18.4    False  
               2019                  11.5    False  
               2020                  17.0    False  
               2021                  15.8    False  
               2022                  13.7    False  
               2023                  20.4    False  
               2024                  15.8    False  
               2025                  18.9    False

No year flagged: all images have less than 30% of masked pixels.


The masked pixels are displayed in **magenta** on the true-colour images, to check that the SCL mask really follows the clouds (and does not mask bright snow by mistake).

In [ ]:
MASK_COLOR = [1, 0, 1]  # magenta


def show_mask_grid(name, gain=1.5, ncols=3):
    """True-colour image of every year with the masked pixels in magenta."""
    years = list(images[name])
    nrows = int(np.ceil(len(years) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
    for ax in axes.flat:
        ax.axis("off")
    for ax, year in zip(axes.flat, years):
        rgb = true_color(images[name][year], gain)
        rgb[masks[name][year]] = MASK_COLOR
        ax.imshow(rgb)
        ax.set_title(f"{name} - {selected.loc[(name, year), 'date']} (masked: {masks[name][year].mean():.1%})")
    plt.tight_layout()
    plt.show()


for name in AOIS:
    show_mask_grid(name)

### Observations (2017–2025, selected dates)

- **Very few masked pixels:** thanks to the selection rule of section 3, clouds cover at most 4.5 % of an image (Khumbu 2020) and no year is flagged.
- **Topographic shadows (SCL 2) are not masked:** they cover 12 to 28 % of the images, because the sun is low in November. Masking them would remove a large part of the north faces every year. They are kept, but they are a known limitation: in deep shadow, the NDSI of snow can drop below the threshold.
- **2025 remains snowier than the other years:** even on the least snowy clear date (22 Nov 2025), 67 % of the Khumbu area is snow/ice (32–42 % the other years) and 56 % of the Imja Tsho area (10–19 %). A heavy snowfall in early October 2025 left fresh snow over the whole window. The snow/ice area of 2025 is therefore **not comparable** with the other years and should be excluded from the glacier trend (section 6).